## Building a model that provides top 5 movie recommendations to a user, based on their ratings of other movies.

This model is a content-based filtering recommendation system that will recommend movies to the user based on what they had liked before.

- Import libraries
- Load data
- Cleaning
- EDA
- Feature engineering
- Baseline
- Modeling
- Evaluation
- Final recommendations
- Conclusion

In [27]:
# importing the libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [28]:
# Loading the datasets
links = pd.read_csv('data/links.csv')
movies = pd.read_csv('data/movies.csv')
tags = pd.read_csv('data/tags.csv')
ratings = pd.read_csv('data/ratings.csv')

## Understanding the dataset

In [29]:
#inspecting the datasets
links.head()

,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0


In [30]:
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


Movie dataset has three columns only: movieId, title and genres

In [31]:
tags.head()

,userId,movieId,tag,timestamp
0,2,60756,funny,1445714994
1,2,60756,Highly quotable,1445714996
2,2,60756,will ferrell,1445714992
3,2,89774,Boxing story,1445715207
4,2,89774,MMA,1445715200


Tags dataset has four columns: userId, movieId, tag and timestamp

In [32]:
ratings.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


Ratings dataset has four columns: userId, movieId, rating, timestamp

In [33]:
# checking the size our datasets
links.shape

(9742, 3)

In [34]:
links.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9742 entries, 0 to 9741
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   movieId  9742 non-null   int64  
 1   imdbId   9742 non-null   int64  
 2   tmdbId   9734 non-null   float64
dtypes: float64(1), int64(2)
memory usage: 228.5 KB


The links dataset has **no** missing values. It also has 9742 rows and  three columns

In [35]:
movies.shape

(9742, 3)

In [36]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9742 entries, 0 to 9741
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   movieId  9742 non-null   int64 
 1   title    9742 non-null   object
 2   genres   9742 non-null   object
dtypes: int64(1), object(2)
memory usage: 228.5+ KB


Movie dataset has also no null values. The dataet has 9471 rows and three columns

In [37]:
ratings.shape

(100836, 4)

In [38]:
ratings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100836 entries, 0 to 100835
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     100836 non-null  int64  
 1   movieId    100836 non-null  int64  
 2   rating     100836 non-null  float64
 3   timestamp  100836 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 3.1 MB


Ratings dataset has also no null values. The dataset has 100836 rows columns and 3 columns.

In [39]:
tags.shape

(3683, 4)

In [40]:
tags.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3683 entries, 0 to 3682
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   userId     3683 non-null   int64 
 1   movieId    3683 non-null   int64 
 2   tag        3683 non-null   object
 3   timestamp  3683 non-null   int64 
dtypes: int64(3), object(1)
memory usage: 115.2+ KB


The dataset has 3683 rows and 4 columns

In [41]:
# checking for duplicates
print("Duplicate rows links:" , links.duplicated().sum())
print("Duplicate rows  movies:" , movies.duplicated().sum())
print("Duplicate rows ratings:" , ratings.duplicated().sum())
print("Duplicate rows tags:" , ratings.duplicated().sum())


Duplicate rows links: 0
Duplicate rows  movies: 0
Duplicate rows ratings: 0
Duplicate rows tags: 0


Changing the timestamps into readable formats

In [42]:
pd.to_datetime(ratings["timestamp"], unit="s")

0        2000-07-30 18:45:03
1        2000-07-30 18:20:47
2        2000-07-30 18:37:04
3        2000-07-30 19:03:35
4        2000-07-30 18:48:51
                 ...        
100831   2017-05-03 21:53:22
100832   2017-05-03 22:21:31
100833   2017-05-08 19:50:47
100834   2017-05-03 21:19:12
100835   2017-05-03 21:20:15
Name: timestamp, Length: 100836, dtype: datetime64[ns]

In [43]:
pd.to_datetime(tags["timestamp"], unit="s")

0      2015-10-24 19:29:54
1      2015-10-24 19:29:56
2      2015-10-24 19:29:52
3      2015-10-24 19:33:27
4      2015-10-24 19:33:20
               ...        
3678   2007-02-11 22:46:59
3679   2007-03-08 22:18:54
3680   2017-05-03 20:39:44
3681   2017-05-03 20:39:38
3682   2017-05-03 20:44:30
Name: timestamp, Length: 3683, dtype: datetime64[ns]

## Exploratory Data analysis

In [44]:
links.columns

Index(['movieId', 'imdbId', 'tmdbId'], dtype='object')

In [45]:
movies.columns

Index(['movieId', 'title', 'genres'], dtype='object')

In [46]:
ratings.columns

Index(['userId', 'movieId', 'rating', 'timestamp'], dtype='object')

In [47]:
tags.columns

Index(['userId', 'movieId', 'tag', 'timestamp'], dtype='object')

In [48]:
# The four datasets, we have a common column(movieId) that we can use for merging the datasets.

merged_df = (
    movies
    .merge(links, on = "movieId", how="left")
    .merge(ratings, on = "movieId", how = "left")
    .merge(tags, on="movieId", how="left")
)

In [49]:
merged_df.head()

,movieId,title,genres,imdbId,tmdbId,userId_x,rating,timestamp_x,userId_y,tag,timestamp_y
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,114709,862.0,1.0,4.0,964982703.0,336.0,pixar,1.139046e+09
1,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,114709,862.0,1.0,4.0,964982703.0,474.0,pixar,1.137207e+09
2,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,114709,862.0,1.0,4.0,964982703.0,567.0,fun,1.525286e+09
3,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,114709,862.0,5.0,4.0,847434962.0,336.0,pixar,1.139046e+09
4,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,114709,862.0,5.0,4.0,847434962.0,474.0,pixar,1.137207e+09


All the datasets have been joined and there is no missing value.

In [56]:
merged_df["timestamp_x"] = pd.to_datetime(
    merged_df["timestamp_x"],
    unit="s"
)

In [57]:
merged_df["timestamp_y"] = pd.to_datetime(
    merged_df["timestamp_y"],
    unit="s"
)

In [58]:
merged_df.head()

,movieId,title,genres,imdbId,tmdbId,userId_x,rating,timestamp_x,userId_y,tag,timestamp_y
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,114709,862.0,1.0,4.0,2006-02-04 09:36:04,336.0,pixar,2006-02-04 09:36:04
1,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,114709,862.0,1.0,4.0,2006-01-14 02:47:05,474.0,pixar,2006-01-14 02:47:05
2,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,114709,862.0,1.0,4.0,2018-05-02 18:33:33,567.0,fun,2018-05-02 18:33:33
3,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,114709,862.0,5.0,4.0,2006-02-04 09:36:04,336.0,pixar,2006-02-04 09:36:04
4,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,114709,862.0,5.0,4.0,2006-01-14 02:47:05,474.0,pixar,2006-01-14 02:47:05
